In [ ]:
from forecastlib import analysis as fa



In [ ]:
runs = fa.load_runs("logs/experiment_1")   # TensorBoard events, or metrics.csv if there are none
lstm = runs[runs.model == "LSTM"]          # filtering is plain pandas


## Metrics

In [ ]:
#fa.varying_hparams(runs)
#fa.summary(runs, ["nse", "kge"]).sort_values("nse")   # one row per run: hparams + metrics averaged over the steps


In [ ]:
fig, data = fa.plot_horizon(runs, ["nse","kge","mse"], split=["val", "test"], by=["model", "diff"], band="std",return_data=True)
fig


## Forecasts

In [ ]:
#fa.plot_hparam(runs, "nse", "learning_rate", color="model", log_x=True)

fc = fa.predict(lstm, start="2022-02-18", end="2022-02-25", label="num_layers")
#data = fa.load_data(lstm)    # the runs' data file, indexed by time
data = fa.load_data(lstm, unshift={"yw_reinbek": 48})



In [ ]:
fa.plot_issue(fc, "2022-02-21 10:00")      # observed history + each run's forecast from that time
##fa.plot_lead(fc, [1, 24, 48])              # observed vs forecast at fixed lead times

In [ ]:
fa.plot_forecasts(fc, hours=[0, 12], every=2)   # 00:00 and 12:00 runs, every other one


In [ ]:

fa.plot_issue(fc, "2022-02-21 10:00", secondary=data[["yw_reinbek", "prsn_reinbek"]])

In [ ]:
from forecastlib.analysis import Analysis
import pandas as pd

class MyEnsemble(Analysis):
    def load_observations(self):              # optional, default: the CSV from the hparams
        return pd.read_parquet("data/bille.parquet")          # indexed by time

    def load_members(self, issue_time):       # long: member, valid_time, <forecast columns>
        return read_members(issue_time)

analysis = MyEnsemble("logs/experiment_1").filter("model == 'LSTM'")
fc = analysis.predict(start="2022-02-18", end="2022-02-20", hours=12, label="num_layers")
analysis.plot_forecasts(fc, secondary="yw_reinbek")


